# Multiclass classification using FeedForward Neural Networks
Implementation with Pytorch, testing on the MNIST dataset.


Pyhton 3.12.0

miriamzara@MacBook-Pro-di-Miriam MCP % python3.12 -m pip install torch 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --- Data ---
transform = transforms.Compose([transforms.ToTensor(), lambda x: x.view(-1)])  
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform) 
test_set  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=1000)

# --- Model ---
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # input:                28x28 = 786 neurons
        # first hidden layer:   256         neurons
        # output layer:         10          neurons
        self.fc1 = nn.Linear(28*28, 256)    # creates the trainable weights (256, 28*28) and bias (256) 
        self.fc2 = nn.Linear(256, 10)       # same, creates the trainable parameters of the layer

    def forward(self, x):
        # x: input vector
        x = torch.relu(self.fc1(x)) # computes output of the 1 hidden layer, using ReLu activation
        return self.fc2(x)          # computes the input to the output layer - no activation is needed

model = Net()                               # instantiation of the network

# --- Training setup ---
criterion = nn.CrossEntropyLoss()           # definition of the loss function
optimizer = optim.Adam(model.parameters())  # definition of the otimizer
                                            # takes as argument all the parameters of the model
                                            # which are defined in the __init__() method of class Net()

# --- Train ---
for epoch in range(3):
    for x, y in train_loader:
        optimizer.zero_grad()               # initializes gradients to zero
        loss = criterion(model(x), y)       # compute loss, comparing the current output of
                                            # the model, model(x), and the real label, y.
        loss.backward()                     # loss is not just the vector of losses. It is
                                            # a tensor that tracks the entire computation graph
                                            # that produced it. 
                                            # so you can call directly .backward() on it.
                                            # this method crosses the graph from bottom to top
                                            # computing the gradients. These gradients
                                            # are stored as attributes in the parameter objects

        optimizer.step()                    # reads the gradients and performs the update
    print(f"Epoch {epoch+1} complete")

# --- Test ---
correct = 0
total = 0
model.eval()                                # switch to evaluation mode. for instance, 
                                            # deactivate dropout - if used in the first place. 
                                            # In this very simple network, it is useless. 

with torch.no_grad():                       # avoid the automatic tracking of gradients
                                            # this saves a lot of memory and time, since you do not need it.
                                            # without no_grad(), the call model(x) would automatically
                                            # produce the computational graph.
    for x, y in test_loader:
        logits = model(x)                   # outputs of the output layer
        pred = logits.argmax(1)             # takes the class with the maximum logit 
        correct += (pred == y).sum().item()
        total += y.size(0)

print("Accuracy:", correct / total)

100%|██████████| 9.91M/9.91M [00:05<00:00, 1.82MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 175kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.53MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.88MB/s]


Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Accuracy: 0.9698


Comments on the code:

1. torchvision.transforms contains routines for common image transformations, including cropping, padding, rotation, color to grayscale, brightness boosting and so on. Transformations can be chained together using transforms.Compose(). Custom transformations can also applied.
2. transforms.ToTensor() Converts a PIL Image or numpy.ndarray (H x W x C) in the range [0, 255] to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0]. PIL (Pillow, previously Python Imaging Library) is a specific format for images. 
3. x.view() reshapes a tensor. Example: 

```{python}
x = torch.randn(3, 4, 2) # total = 24 elements
x.view(12, 2) # new shape: (12,2)
x.view(4, -1) # infers the second dimension automatically, new shape: (4, 6)
x.view(-1) # infers the first dimension automatically, new shape: (24,)
```
In this usage, x.view() takes as input the image, which is a 3d tensor (height, width, n channels) and flattens it to a 1D array.

4. DataLoader() provides an iterable over the dataset. The batch size parameter specifies how many samples should be loaded at once. Pytorch methods support batch processing. This mean that in the evaluating phase you can feed the whole test dataset at once to the network - if not too big - or in batches, in general. For the training phase, the batch size is used for stochastic gradient descent computation. Smaller batch -> noisier gradient descent steps, less computationally expensive. Larger batch -> less noise, more expensive, can get stuck in local minima. 